# Optimisation de systèmes énergétiques avec PyPSA

Ce notebook rassemble cinq études de cas progressives :
1. **Planification à un nœud** (*one-node planning*) ;
2. **Courbes de dépistage** (*screening curves*) : l’équilibre de long terme « à la main » puis avec PyPSA ;
3. **Modélisation pour générer des alternatives** (*MGA*) ;
4. **Planification à trois nœuds** (*three-node planning*) ;
5. **Planification myope** (*myopic planning*), exercice de synthèse sur plusieurs périodes.

> **Prérequis.** Environnement Python avec `pypsa`, `pandas`, `numpy` et `matplotlib`. Certaines données d’exemple sont téléchargées depuis les ressources PyPSA/TU Berlin : une connexion Internet est donc nécessaire.

## 0. Préparation commune

On commence par importer les bibliothèques. On ajoute aussi une petite fonction utilitaire `copier_reseau`, utile pour comparer des variantes sans modifier le réseau de référence. Les modèles utilisent des puissances en MW, des énergies en MWh et des coûts cohérents avec les données de chaque exemple.

In [ ]:
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pypsa

logging.getLogger("pypsa").setLevel(logging.ERROR)    # messages d'information masqués (on relance beaucoup de calculs)
logging.getLogger("linopy").setLevel(logging.WARNING)  # on garde les alertes, par exemple en cas d'infaisabilité
pypsa.options.params.optimize.log_to_console = False  # sorties solveur masquées (on relance beaucoup de calculs)

print("Version de PyPSA :", pypsa.__version__)


def copier_reseau(n):
    """Copie un réseau, y compris s'il a déjà été optimisé.

    Dans les versions récentes de PyPSA, `n.copy()` refuse de copier un réseau résolu
    tant que le modèle du solveur est attaché : on le détache d'abord.
    """
    if getattr(n, "model", None) is not None:
        n.model.solver_model = None
    return n.copy()

---

# Exercice 1 — Planification à un nœud

## Objectif

Déterminer les capacités optimales de production et de stockage permettant de satisfaire une demande électrique variable au moindre coût. Le réseau est représenté par **un seul nœud** : les contraintes de transport d’électricité ne sont donc pas représentées.

Le modèle choisit simultanément les investissements (CAPEX annualisés) et l’exploitation (OPEX). Les séries temporelles sont échantillonnées toutes les trois heures pour réduire le temps de calcul.

### 1.1 Préparation des données et du réseau

Pour rester proche de l’exemple fourni, on utilise le jeu de données `model.energy` exposé par PyPSA. La préparation des données est ainsi réduite à un appel et les hypothèses restent regroupées dans le modèle d’exemple.

In [ ]:
n = pypsa.examples.model_energy()
n

### 1.2 Résolution du modèle

Le réseau contient déjà la demande, les profils de production renouvelable et les technologies de stockage. L’optimisation détermine les capacités installées et les dispatchs horaires.

In [ ]:
n.optimize()

### 1.3 Analyse des résultats

Les statistiques de PyPSA permettent de comparer les coûts d’investissement et d’exploitation, les capacités optimales et les bilans énergétiques. Les capacités sont affichées en GW (et en GWh pour les stockages).

In [ ]:
### DEBUT SOLUTION
couts = pd.concat([n.statistics.capex(), n.statistics.opex()], axis=1)
couts.columns = ["Investissement annualisé", "Exploitation"]
display(couts / 1e9)
display(n.statistics.optimal_capacity().div(1e3))
display(n.statistics.energy_balance(bus_carrier="electricity").sort_values().div(1e6))
n.statistics.energy_balance.plot.area(linewidth=0, bus_carrier="electricity")
plt.show()
### FIN SOLUTION


**À retenir.** Le modèle arbitre entre technologies en fonction de leurs coûts, de leurs profils de disponibilité et de la flexibilité offerte par le stockage. Une capacité renouvelable importante ne signifie pas qu’elle produit à pleine puissance à chaque instant.

### 1.4 Pour explorer : prix, recouvrement des coûts et valeur de la flexibilité

#### 1.4.a Le prix marginal de l'électricité

Rappel de cours : le prix marginal à l'instant $t$ est la variable duale de la contrainte d'équilibre offre-demande, c'est-à-dire le coût d'**un MWh supplémentaire** consommé à cet instant. Ici, le modèle investit **et** exploite : on peut donc se demander si les prix couvrent les coûts totaux du système.

In [ ]:
# TODO : calculer (i) le prix moyen pondéré par la demande, (ii) le coût moyen du système (coût total / énergie
# consommée), (iii) la part du temps où le prix est nul, puis tracer la courbe de durée des prix.
### DEBUT SOLUTION
prix = n.buses_t.marginal_price["electricity"]
poids = n.snapshot_weightings.generators
demande = n.loads_t.p.sum(axis=1)          # MW
energie = (demande * poids).sum()          # MWh sur la période simulée

prix_moyen = (prix * demande * poids).sum() / energie
cout_total = n.statistics.capex().sum() + n.statistics.opex().sum()
cout_moyen = cout_total / energie
part_prix_nul = poids[prix.abs() < 1e-6].sum() / poids.sum()

print(f"Prix moyen pondéré par la demande : {prix_moyen:8.2f} €/MWh")
print(f"Coût moyen du système             : {cout_moyen:8.2f} €/MWh")
print(f"Part du temps à prix nul          : {100 * part_prix_nul:8.1f} %")

fig, ax = plt.subplots(figsize=(7, 3))
prix.sort_values(ascending=False).reset_index(drop=True).plot(ax=ax)
ax.set_xlabel("Pas de temps classés par prix décroissant")
ax.set_ylabel("€/MWh")
ax.set_title("Courbe de durée des prix")
plt.show()
### FIN SOLUTION

**Éléments de réponse** *(cellule de correction)*

- Le **prix moyen pondéré par la demande est égal au coût moyen du système** (à l'arrondi près). C'est la dualité forte : dans un modèle linéaire où l'on peut investir librement avec des coûts proportionnels à la capacité (rendements constants), *chaque technologie couvre exactement ses coûts avec les prix marginaux* (profit nul). Vous le démontrerez explicitement à l'exercice 2.
- Un prix **nul** (ou quasi nul) apparaît quand l'offre renouvelable est excédentaire : un MWh de demande en plus ne coûte rien. Un prix **élevé** apparaît quand il faut déstocker ou utiliser la technologie la plus chère pour passer la pointe.
- La courbe de durée des prix a donc une forme en « marches » : chaque palier correspond à une technologie (ou à une valeur de stock) qui fixe le prix.

#### 1.4.b Écrêtement : produire moins que ce qui est disponible

Le modèle peut **écrêter** (curtailment) l'éolien et le solaire : produire moins que la disponibilité fournie par `p_max_pu`. Cela peut sembler paradoxal pour des capacités dimensionnées par l'optimisation.

In [ ]:
# TODO : calculer, pour le solaire et l'éolien, la part de l'énergie disponible qui est écrêtée (en %).
### DEBUT SOLUTION
ecretement = n.statistics.curtailment(components="Generator")   # MWh perdus
production = n.statistics.supply(components="Generator")         # MWh produits
part_ecretee = (100 * ecretement / (production + ecretement)).round(1)
display(part_ecretee.reindex(["solar", "wind"]).rename("Part écrêtée (%)"))
### FIN SOLUTION

**Éléments de réponse** *(cellule de correction)*

Un écrêtement non nul est **optimal** : dimensionner la capacité pour qu'aucune énergie ne soit jamais perdue serait trop cher. Il vaut mieux accepter de perdre une petite part de la production lors des rares moments d'excédent plutôt que d'ajouter du stockage ou de la demande flexible dont le coût dépasse la valeur de l'énergie sauvée. L'écrêtement mesure en quelque sorte le « prix à payer » pour avoir un système renouvelable bon marché.

#### 1.4.c Sensibilités : que se passe-t-il si les hypothèses changent ?

On définit une petite fonction qui résout une **variante** du réseau de référence : coût du solaire modifié, ou technologies de stockage supprimées. Le but est de quantifier la **valeur de la flexibilité** (stockage) et la sensibilité du mix aux coûts.

> 💡 **Avant d'exécuter, prédisez** : (1) si le solaire coûte deux fois moins cher, que devient la capacité éolienne ? (2) Quelle suppression augmente le plus le coût : celle de la batterie ou celle de l'hydrogène ?

In [ ]:
def resoudre_variante(n_ref, facteur_cout_solaire=1.0, retirer=()):
    """Résout une variante du réseau `n_ref`.

    facteur_cout_solaire : multiplicateur du coût d'investissement du solaire.
    retirer : liste de tuples (composant, nom) à supprimer, ex. [("StorageUnit", "battery storage")].
    """
    m = copier_reseau(n_ref)
    m.generators.loc["solar", "capital_cost"] *= facteur_cout_solaire
    for composant, nom in retirer:
        m.remove(composant, nom)
    m.optimize()
    return m


scenarios = {
    "référence": {},
    "solaire -50 %": {"facteur_cout_solaire": 0.5},
    "solaire +100 %": {"facteur_cout_solaire": 2.0},
    "sans batterie": {"retirer": [("StorageUnit", "battery storage")]},
    "sans hydrogène": {"retirer": [("Link", "electrolysis"), ("Link", "turbine"), ("Store", "hydrogen storage")]},
}

In [ ]:
# TODO : résoudre chaque scénario et construire un tableau (capacités solaire/éolien en GW, coût total en G€).
### DEBUT SOLUTION
resultats = {}
for nom, options in scenarios.items():
    m = n if nom == "référence" else resoudre_variante(n, **options)
    cap = m.generators.p_nom_opt
    resultats[nom] = {
        "Solaire (GW)": cap.get("solar", 0) / 1e3,
        "Éolien (GW)": cap.get("wind", 0) / 1e3,
        "Coût total (G€)": (m.statistics.capex().sum() + m.statistics.opex().sum()) / 1e9,
    }
tableau = pd.DataFrame(resultats).T.round(2)
display(tableau)

tableau[["Solaire (GW)", "Éolien (GW)"]].plot.bar(figsize=(8, 3.5), ylabel="GW", rot=20, title="Mix de production selon le scénario")
plt.show()
### FIN SOLUTION

**Éléments de réponse** *(cellule de correction)*

- Quand le solaire devient moins cher, il **remplace partiellement l'éolien** ; l'inverse se produit s'il devient plus cher. Les deux technologies sont substituables mais leurs profils temporels diffèrent (jour/nuit, saisons), donc le remplacement n'est jamais 1 pour 1.
- Supprimer un moyen de flexibilité **augmente toujours le coût** (on a retiré une option, le problème est plus contraint). L'écart de coût avec la référence est la *valeur du système de stockage retiré*. Comparez l'effet de la batterie (flexibilité de court terme) et de l'hydrogène (stockage de longue durée, qui gère les périodes de faible production prolongée).
- Remarque méthodologique : un modèle d'optimisation donne des **ordres de grandeur robustes** et des tendances, pas des prédictions exactes : testez toujours la sensibilité aux hypothèses de coûts.

---

# Exercice 2 — Courbes de dépistage (*screening curves*) et équilibre de long terme

## Objectifs

Calculer l'**équilibre de long terme** du parc de production pour une courbe monotone de charge donnée et un ensemble de technologies d'investissement, puis **retrouver numériquement** le résultat théorique du cours (courbes de dépistage). Cet exercice est le « cas d'école » qui explique pourquoi le modèle de l'exercice 1 choisit tel ou tel mix.

Vous allez :
1. prédire le parc optimal **à la main** à partir des courbes de dépistage ;
2. le vérifier avec PyPSA ;
3. interpréter les **prix** (valeur duale) et la **condition de profit nul** ;
4. expérimenter : coût d'investissement, prix du carbone, nouvelle technologie, plafond de prix.

Cet exercice reprend l'exemple `generation-investment-screening-curve` de la documentation PyPSA. Pour la théorie, voir [ce cours](https://nworbmot.org/courses/es-25/es-10-investment.pdf).

### 2.1 Théorie : la courbe de dépistage

Une centrale de capacité 1 MW qui fonctionne pendant une fraction $x \in [0,1]$ de l'année coûte, par an :

$$C(x) = c + m \, x$$

où $c$ est le coût d'investissement annualisé (par MW) et $m$ le coût marginal (par MWh, l'année étant normalisée à 1). C'est une **droite** dont l'ordonnée à l'origine est $c$ et la pente $m$. Pour une durée de fonctionnement $x$ donnée, la technologie optimale est celle dont la droite est **la plus basse**.

Les coûts sont choisis pour obtenir des résultats simples :

| Technologie | $c$ (investissement) | $m$ (marginal) |
|---|---|---|
| charbon | 15 | 2 |
| gaz | 10 | 12 |
| délestage | 0 | 1012 |

Le **délestage** (*load shedding*) est une « centrale » fictive sans investissement mais très chère : elle représente le coût de la demande non servie (VOLL, *value of lost load*). La charge est la courbe monotone $1000 - 1000\,z$ MW pour $z \in [0,1]$ (la fraction du temps où la charge dépasse la valeur correspondante).

In [ ]:
generateurs = {
    "coal": {"m": 2, "c": 15},
    "gas": {"m": 12, "c": 10},
    "load-shedding": {"m": 1012, "c": 0},
}

x = np.linspace(0, 1, 101)
courbes = pd.DataFrame({k: v["c"] + x * v["m"] for k, v in generateurs.items()}, index=x)
ax = courbes.plot(ylim=[0, 50], title="Courbes de dépistage", figsize=(7, 4))
ax.set_xlabel("Fraction de l'année où la centrale fonctionne, x")
ax.set_ylabel("Coût annuel par MW")
plt.show()

> ✍️ **À faire sur papier avant de coder.**
> 1. Calculez les abscisses des **deux intersections** des courbes (charbon/gaz et gaz/délestage).
> 2. En déduire, à l'aide de la courbe monotone de charge $1000-1000z$, la **capacité optimale** de chaque technologie (MW).

**Éléments de réponse** *(cellule de correction)*

- Gaz/charbon : $10 + 12x = 15 + 2x \Rightarrow x = 0{,}5$. Délestage/gaz : $0 + 1012x = 10 + 12x \Rightarrow x = 0{,}01$.
- Le charbon est optimal pour les durées de fonctionnement supérieures à 50 % (**base**), le gaz entre 1 % et 50 % (**semi-base**), le délestage en dessous de 1 % (**pointe extrême**).
- La charge dépasse 500 MW pendant 50 % du temps : charbon = **500 MW**. Elle dépasse 10 MW pendant 99 % du temps, donc le gaz couvre de 500 à 990 MW : gaz = **490 MW**. Le délestage couvre les 10 derniers MW : **10 MW**.

### 2.2 Le modèle PyPSA

Le réseau a un seul bus. Les 1001 « instants » représentent $z \in [0,1]$, chacun avec un poids $1/1001$ : la somme des poids vaut 1 (année normalisée), ce qui garantit que $c$ est bien un coût par MW et par an. Toutes les capacités sont **extensibles** (`p_nom_extendable=True`).

La fonction ci-dessous (fournie) construit et résout le modèle pour un dictionnaire de technologies : vous la réutiliserez dans toute la suite.

In [ ]:
def construire_reseau(generateurs, n_snap=1001):
    """Un bus, une charge monotone 1000 - 1000 z, et un générateur extensible par technologie."""
    n = pypsa.Network()
    n.snapshots = np.linspace(0, 1, n_snap)
    n.snapshot_weightings = n.snapshot_weightings / n_snap   # somme des poids = 1
    n.add("Bus", name="bus")
    n.add("Load", name="charge", bus="bus", p_set=1000 - 1000 * n.snapshots.values)
    for nom, p in generateurs.items():
        n.add(
            "Generator", name=nom, bus="bus", p_nom_extendable=True,
            marginal_cost=p["m"], capital_cost=p["c"],
        )
    return n


def resoudre(generateurs):
    n = construire_reseau(generateurs)
    n.optimize()
    return n

In [ ]:
# TODO : résoudre le modèle, afficher les capacités optimales et comparer avec votre prédiction.
# Tracer la courbe monotone de charge.
### DEBUT SOLUTION
n_sc = resoudre(generateurs)
display(n_sc.generators.p_nom_opt.round(2))

n_sc.loads_t.p_set.plot.area(title="Courbe monotone de charge", ylabel="MW", xlabel="z", figsize=(7, 3), legend=False)
plt.show()
### FIN SOLUTION

**Éléments de réponse** *(cellule de correction)*

On retrouve **charbon = 500 MW, gaz = 490 MW, délestage = 10 MW** : l'optimisation reproduit exactement le résultat graphique. Le délestage n'est utilisé que 1 % du temps : c'est son coût marginal (1012) qui détermine le prix pendant ces heures, mais il n'est pas « cher » au sens du coût annuel puisqu'on s'en sert très peu.

### 2.3 Les prix : courbe de durée des prix

Le prix marginal est la valeur duale de l'équilibre offre-demande. Comparez-le aux coûts marginaux des technologies.

In [ ]:
# TODO : tracer la courbe de durée des prix et compter le nombre d'instants pour chaque valeur de prix.
### DEBUT SOLUTION
prix = n_sc.buses_t.marginal_price["bus"]
prix.plot(title="Courbe de durée des prix", xlabel="z", ylabel="€/MWh", figsize=(7, 3))
plt.show()
display(prix.round(2).value_counts())
### FIN SOLUTION

**Éléments de réponse** *(cellule de correction)*

Les prix valent le **coût marginal de la technologie marginale** : 1012 pour $z<0{,}01$ (délestage), 12 pour $0{,}01<z<0{,}5$ (gaz), 2 pour $z>0{,}5$ (charbon). Trois instants isolés ont des valeurs « bizarres » :
- **22 = 12 + 10** à $z=0{,}01$ : augmenter un peu la charge en ce point obligerait à **investir** dans du gaz (coût marginal 12 + investissement 10) à la place du délestage ;
- **7 = 12 + 10 − 15** à $z=0{,}5$ : on remplace du charbon par du gaz (on économise l'investissement charbon 15) ;
- **0** à $z=1$ : la charge est nulle, aucune centrale ne tourne : le prix est à sa borne inférieure (cas dégénéré où le prix pourrait valoir n'importe quelle valeur entre 0 et 2).

Ces points correspondent à **1 instant sur 1001** (poids négligeable) : ce sont des artefacts de discrétisation, pas des phénomènes physiques.

### 2.4 La condition de profit nul

À l'équilibre de long terme (rendements constants, concurrence parfaite), **chaque technologie couvre exactement ses coûts** au prix marginal. Vérifions-le : pour chaque technologie, on calcule le coût total (investissement + exploitation), le revenu (production × prix marginal) et le profit.

In [ ]:
def bilan_economique(n):
    """Coût, revenu et profit annuels de chaque générateur (unités du modèle)."""
    poids = n.snapshot_weightings.objective
    prix = n.buses_t.marginal_price["bus"]
    energie = n.generators_t.p.mul(poids, axis=0).sum()
    cout_inv = n.generators.p_nom_opt * n.generators.capital_cost
    cout_exp = energie * n.generators.marginal_cost
    revenu = n.generators_t.p.mul(prix, axis=0).mul(poids, axis=0).sum()
    bilan = pd.DataFrame({
        "Capacité (MW)": n.generators.p_nom_opt,
        "Coût inv.": cout_inv, "Coût exploit.": cout_exp, "Revenu": revenu,
    })
    bilan["Profit"] = bilan["Revenu"] - bilan["Coût inv."] - bilan["Coût exploit."]
    return bilan.round(2)

In [ ]:
### DEBUT SOLUTION
display(bilan_economique(n_sc))
### FIN SOLUTION

**Éléments de réponse** *(cellule de correction)*

Les profits sont **tous nuls**, y compris pour le gaz et le délestage. Le gaz, qui ne tourne pas en base, récupère son coût d'investissement grâce à la **rente de rareté** : pendant 1 % du temps le prix vaut 1012 au lieu de 12, soit un surplus de $(1012-12)\times 0{,}01 = 10$ par MW, exactement son coût d'investissement. Le charbon, lui, touche des **rentes inframarginales** quand le prix est fixé par le gaz (12 > 2).

### 2.5 Et si on figeait les capacités ? Court terme vs long terme

On prend les capacités de l'équilibre de long terme, on **interdit l'expansion**, puis on relance l'optimisation d'exploitation seule. Les prix sont-ils les mêmes ?

In [ ]:
### DEBUT SOLUTION
n_ct = copier_reseau(n_sc)
n_ct.generators["p_nom_extendable"] = False
n_ct.generators["p_nom"] = n_ct.generators["p_nom_opt"]
n_ct.optimize()

prix_ct = n_ct.buses_t.marginal_price["bus"]
ecart = (prix_ct - prix).abs()
print("Nombre d'instants où le prix diffère :", int((ecart > 1e-6).sum()), "sur", len(ecart))
display(ecart[ecart > 1e-6].round(2))
### FIN SOLUTION

**Éléments de réponse** *(cellule de correction)*

Les prix de court et de long terme sont **identiques sauf en quelques instants singuliers** : ici 2 instants sur 1001, ceux des intersections vus en 2.3 ($z=0{,}01$ avec un écart de 10, le coût d'investissement du gaz ; $z=0{,}5$ avec un écart de 5), où le prix de long terme intégrait un coût d'investissement. C'est la propriété de l'équilibre : le marché de court terme (exploitation) reproduit les prix de long terme *quand les capacités sont optimales*. Cette équivalence **n'est pas générale** : elle se dégrade par exemple avec beaucoup de stockage et de décalage temporel de l'énergie (voir exercices 1 et 4).

### 2.6 Explorer 1 : le coût d'investissement du charbon

> 💡 **Prédisez** : à partir de quelle valeur de $c_{charbon}$ le charbon disparaît-il du mix ? Et à partir de quelle valeur le gaz disparaît-il (hors délestage) ? *(Indice : regardez les ordonnées à l'origine et la valeur des droites en $x=1$.)*

In [ ]:
# TODO : faire varier c_charbon de 0 à 30 et tracer les capacités optimales de chaque technologie.
### DEBUT SOLUTION
couts_charbon = np.arange(0, 31, 2)
capacites = {}
for c_charbon in couts_charbon:
    g = {k: dict(v) for k, v in generateurs.items()}
    g["coal"]["c"] = c_charbon
    capacites[c_charbon] = resoudre(g).generators.p_nom_opt

df_cap = pd.DataFrame(capacites).T
df_cap.plot.area(figsize=(7, 3.5), xlabel="Coût d'investissement du charbon, c", ylabel="Capacité optimale (MW)")
plt.show()
### FIN SOLUTION

**Éléments de réponse** *(cellule de correction)*

Trois régimes :
- $c_{charbon} \le 10$ : le charbon est moins cher **en investissement et en exploitation** que le gaz, il domine (le gaz disparaît) ;
- $10 < c_{charbon} < 20$ : mix charbon + gaz, la part du charbon diminue quand $c$ augmente ;
- $c_{charbon} \ge 20$ : même en base ($x=1$) le charbon coûte $c+2 \ge 22$, soit autant que le gaz ($10+12=22$) : il disparaît.

Le passage d'un régime à l'autre est **continu** (capacités affines par morceaux) mais change brutalement de pente : c'est typique d'un problème linéaire (changement de base optimale).

### 2.7 Explorer 2 : un prix du carbone

On ajoute un prix du CO₂ $P$ (€/tCO₂) aux coûts marginaux, avec des facteurs d'émission **illustratifs** : charbon 1,0 tCO₂/MWh, gaz 0,4 tCO₂/MWh. Le coût marginal devient $m + e \cdot P$.

> ✍️ Calculez analytiquement le prix $P^*$ à partir duquel le charbon n'est plus compétitif **même en base** ($x=1$). Vérifiez ensuite avec PyPSA.

In [ ]:
emissions = {"coal": 1.0, "gas": 0.4, "load-shedding": 0.0}   # tCO2/MWh (valeurs illustratives)

def generateurs_avec_carbone(prix_co2):
    return {k: {"c": v["c"], "m": v["m"] + emissions[k] * prix_co2} for k, v in generateurs.items()}

In [ ]:
# TODO : balayer P de 0 à 20 €/t ; tracer les capacités optimales et les émissions annuelles totales.
### DEBUT SOLUTION
prix_co2 = np.arange(0, 21, 1)
cap_co2, emis_co2 = {}, {}
for P in prix_co2:
    m = resoudre(generateurs_avec_carbone(P))
    cap_co2[P] = m.generators.p_nom_opt
    energie = m.generators_t.p.mul(m.snapshot_weightings.objective, axis=0).sum()
    emis_co2[P] = sum(energie[k] * emissions[k] for k in emissions)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
pd.DataFrame(cap_co2).T.plot.area(ax=axes[0], xlabel="Prix du CO₂ (€/t)", ylabel="Capacité (MW)")
pd.Series(emis_co2).plot(ax=axes[1], marker="o", xlabel="Prix du CO₂ (€/t)", ylabel="Émissions (tCO₂, année normalisée)")
plt.tight_layout()
plt.show()
### FIN SOLUTION

**Éléments de réponse** *(cellule de correction)*

- Condition : $15 + 2 + 1{,}0\,P \ge 10 + 12 + 0{,}4\,P \Rightarrow P^* = 5/0{,}6 \approx 8{,}3$ €/tCO₂. Au-delà, le charbon sort **complètement** du mix (la simulation confirme : entre $P=8$ et $P=9$).
- C'est le phénomène de **basculement charbon → gaz** (*fuel switching*) : un prix du carbone modifie l'ordre de mérite et donc le mix d'investissement.
- Les émissions baissent **par paliers/pente** puis se stabilisent : après la sortie du charbon, un prix du carbone plus élevé n'apporte plus rien dans ce modèle (il n'y a plus de technologie bas-carbone disponible). Ajoutez-en une à l'exercice suivant pour voir ce qui change.

### 2.8 Explorer 3 : une nouvelle technologie

Ajoutez une technologie **nucléaire** : $c = 16$, $m = 0{,}5$ (prix du carbone nul).

> ✍️ Prédisez d'abord à quelles durées de fonctionnement le nucléaire est optimal (intersection avec le charbon), puis les nouvelles capacités.

In [ ]:
# TODO : ajouter "nuclear" au dictionnaire, résoudre, afficher les capacités et tracer les nouvelles courbes de dépistage.
### DEBUT SOLUTION
g_nuc = {k: dict(v) for k, v in generateurs.items()}
g_nuc["nuclear"] = {"m": 0.5, "c": 16}

n_nuc = resoudre(g_nuc)
display(n_nuc.generators.p_nom_opt.round(1))

courbes_nuc = pd.DataFrame({k: v["c"] + x * v["m"] for k, v in g_nuc.items()}, index=x)
courbes_nuc.plot(ylim=[0, 30], title="Courbes de dépistage avec nucléaire", figsize=(7, 3.5))
plt.show()
### FIN SOLUTION

**Éléments de réponse** *(cellule de correction)*

Nucléaire/charbon : $16 + 0{,}5x = 15 + 2x \Rightarrow x = 2/3$. Le nucléaire prend donc la **base** (charge supérieure à $1000\times(1-2/3)\approx 333$ MW) : **nucléaire ≈ 333 MW, charbon ≈ 167 MW, gaz = 490 MW, délestage = 10 MW**. Le charbon est repoussé vers la semi-base : la technologie à fort coût fixe et faible coût marginal « déplace » les autres dans l'ordre de mérite. On retrouve l'idée classique : **base = fort $c$/faible $m$, pointe = faible $c$/fort $m$**.

### 2.9 Explorer 4 : un plafond de prix et le problème du « missing money »

Dans la réalité, les prix sont souvent **plafonnés** (réglementation) en dessous du coût de la demande non servie. Conservons les capacités de l'équilibre de long terme, mais calculons ce que gagnerait chaque centrale si le prix était **écrêté** à un plafond.

In [ ]:
# TODO : pour des plafonds de 100, 200, 500 et 1012 €/MWh, calculer le profit par MW installé de chaque technologie.
### DEBUT SOLUTION
poids = n_sc.snapshot_weightings.objective
bilan = bilan_economique(n_sc)
couts_totaux = bilan["Coût inv."] + bilan["Coût exploit."]

profit_par_mw = {}
for plafond in [100, 200, 500, 1012]:
    revenu = n_sc.generators_t.p.mul(prix.clip(upper=plafond), axis=0).mul(poids, axis=0).sum()
    profit_par_mw[plafond] = ((revenu - couts_totaux) / n_sc.generators.p_nom_opt).round(2)

display(pd.DataFrame(profit_par_mw).rename_axis("Plafond de prix (€/MWh)", axis=1))
### FIN SOLUTION

**Éléments de réponse** *(cellule de correction)*

Avec un plafond inférieur à 1012, **toutes les technologies deviennent déficitaires**. Pour le charbon et le gaz, qui tournent à pleine puissance pendant la pénurie, le déficit par MW est identique : $(1012 - \text{plafond}) \times 0{,}01$ (la pénurie dure 1 % du temps). Par exemple, un plafond à 200 donne un déficit d'environ 8,1 par MW. Le délestage est moins touché car il ne produit en moyenne qu'une partie de sa capacité pendant la pénurie.

C'est le problème du **missing money** : la rente de rareté finançait les coûts fixes de l'ensemble du parc. En pratique, il faut alors un **mécanisme de capacité** (rémunération de la puissance disponible) pour que les investisseurs investissent. Ce déficit est justement le montant que ce mécanisme devrait verser.

**Questions de synthèse**
1. Pourquoi la condition de profit nul n'implique-t-elle pas que chaque heure soit « rentable » pour chaque centrale ?
2. Comment un prix du carbone modifie-t-il l'ordre de mérite et le mix d'investissement ?
3. Si vous deviez justifier l'existence d'un mécanisme de capacité, quel résultat de cet exercice utiliseriez-vous ?

---

# Exercice 3 — Modélisation pour générer des alternatives (MGA)

## Objectif

L’optimum économique ne constitue pas nécessairement l’unique solution intéressante pour la planification. La méthode **MGA** explore des solutions alternatives dont le coût reste proche de l’optimum, mais qui favorisent un autre critère (par exemple, réduire la capacité éolienne).

On utilise le même modèle de planification à un nœud. Pour rendre la comparaison pertinente, on calcule d’abord l’optimum de coût, puis on lance MGA avec une marge de coût de 5 %.

### 3.1 Calculer l’optimum économique

In [ ]:
n_mga = pypsa.examples.model_energy()
### DEBUT SOLUTION
n_mga.optimize()
cout_optimal = n_mga.statistics.capex().sum() + n_mga.statistics.opex().sum()
print(f"Coût total optimal : {cout_optimal:,.0f}")
n_opt = copier_reseau(n_mga)  # sauvegarde de l'optimum pour la section 3.4
### FIN SOLUTION


### 3.2 Rechercher une alternative

Le dictionnaire `weights` définit l’objectif alternatif. Ici, on minimise la capacité nominale de la technologie éolienne (`Generator`, variable `p_nom`, générateur nommé `wind`). Le paramètre `slack=0.05` autorise un coût jusqu’à 5 % supérieur à celui de la solution optimale.

In [ ]:
### DEBUT SOLUTION
weights = {"Generator": {"p_nom": {"wind": 1}}}
n_mga.optimize.optimize_mga(slack=0.05, weights=weights, sense="min")
### FIN SOLUTION


### 3.3 Comparer les résultats

In [ ]:
### DEBUT SOLUTION
cout_mga = n_mga.statistics.capex().sum() + n_mga.statistics.opex().sum()
print(f"Coût de la solution MGA : {cout_mga:,.0f}")
print(f"Plafond de coût (5 %) : {cout_optimal * 1.05:,.0f}")
display(n_mga.statistics.optimal_capacity().div(1e3))
display(n_mga.statistics.energy_balance(bus_carrier="electricity").sort_values().div(1e6))
### FIN SOLUTION


**À retenir.** MGA ne cherche pas à améliorer le coût minimal : elle explore des solutions alternatives dans un budget défini. Elle aide à rendre visibles les arbitrages entre technologies et la non-unicité des solutions proches de l’optimum.

### 3.4 Pour explorer : cartographier l'espace des solutions proches de l'optimum

Une seule alternative MGA ne dit pas grand-chose. On va maintenant **encadrer** la capacité de chaque technologie : quelles valeurs minimale et maximale peut-elle prendre si on accepte un surcoût $\varepsilon$ ?

La fonction suivante (fournie) part de l'optimum économique `n_opt` (sauvegardé en 3.1), cherche la solution qui minimise ou maximise la capacité d'une technologie donnée, sous la contrainte $\text{coût} \le (1+\varepsilon)\,\text{coût}_{opt}$, et renvoie les capacités obtenues (en GW).

In [ ]:
def explorer_mga(n_opt, composant, nom, slack, sense):
    """Optimum MGA sur la capacité (p_nom) d'un générateur ou d'un lien. Retourne les capacités en GW."""
    m = copier_reseau(n_opt)
    m.optimize.optimize_mga(slack=slack, weights={composant: {"p_nom": {nom: 1}}}, sense=sense)
    cap = pd.concat([m.generators.p_nom_opt, m.links.p_nom_opt])
    return (cap[["solar", "wind", "electrolysis", "turbine"]] / 1e3).round(2)

#### 3.4.a Un surcoût de 5 % : quatre solutions « extrêmes »

Calculez les solutions qui **minimisent** et **maximisent** la capacité éolienne, puis la capacité solaire, avec `slack=0.05`. Comparez-les à l'optimum.

In [ ]:
# TODO : construire un tableau (lignes : optimum, éolien min, éolien max, solaire min, solaire max ; colonnes : capacités en GW).
### DEBUT SOLUTION
cap_opt = pd.concat([n_opt.generators.p_nom_opt, n_opt.links.p_nom_opt])
cap_opt = (cap_opt[["solar", "wind", "electrolysis", "turbine"]] / 1e3).round(2)

solutions = {"optimum": cap_opt}
for composant, nom, label in [("Generator", "wind", "éolien"), ("Generator", "solar", "solaire")]:
    for sense in ["min", "max"]:
        solutions[f"{label} {sense}"] = explorer_mga(n_opt, composant, nom, slack=0.05, sense=sense)

tableau_mga = pd.DataFrame(solutions).T
display(tableau_mga)
tableau_mga.plot.bar(figsize=(8, 3.5), ylabel="GW", rot=20, title="Solutions à moins de 5 % de surcoût")
plt.show()
### FIN SOLUTION

**Éléments de réponse** *(cellule de correction)*

Pour un surcoût de **5 % seulement**, la capacité éolienne peut varier de manière importante autour de l'optimum (mesurez l'ampleur sur vos propres résultats : en pourcentage de la capacité optimale), et solaire et éolien se compensent en partie : quand l'un monte, l'autre baisse. **L'optimum économique est donc « plat »** : beaucoup de mix très différents coûtent presque la même chose. Ce résultat est essentiel pour la planification : si l'acceptabilité sociale, la disponibilité des sites ou la sécurité d'approvisionnement favorisent un mix particulier, on peut souvent l'obtenir pour un faible surcoût.

#### 3.4.b L'influence de la marge de coût

> 💡 **Prédisez** : si on double la marge (de 5 % à 10 %), l'intervalle de capacités possibles double-t-il exactement ? Plus ? Moins ?

In [ ]:
# TODO : pour des marges de 1 %, 5 % et 10 %, calculer la capacité éolienne minimale et maximale, puis tracer l'intervalle.
### DEBUT SOLUTION
marges = [0.01, 0.05, 0.10]
bornes = {}
for slack in marges:
    bornes[slack] = {
        sense: explorer_mga(n_opt, "Generator", "wind", slack, sense)["wind"] for sense in ["min", "max"]
    }
bornes = pd.DataFrame(bornes).T
bornes["optimum"] = cap_opt["wind"]
display(bornes)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.fill_between([100 * s for s in marges], bornes["min"], bornes["max"], alpha=0.3, label="Plage de capacités possibles")
ax.axhline(cap_opt["wind"], color="k", ls="--", label="Optimum économique")
ax.set_xlabel("Marge de coût acceptée (%)")
ax.set_ylabel("Capacité éolienne (GW)")
ax.legend()
plt.show()
### FIN SOLUTION

**Éléments de réponse** *(cellule de correction)*

La plage s'élargit quand la marge augmente, mais **pas proportionnellement** : la relation est concave (rendements décroissants). Les premiers pourcents de surcoût « ouvrent » beaucoup de flexibilité, car on est près d'un optimum plat ; plus on s'éloigne, plus il devient coûteux de pousser dans une direction.

**Questions de discussion**
1. Que signifierait une plage très étroite pour une technologie ? Et une plage très large ?
2. Comment présenteriez-vous ces résultats à un décideur public qui hésite entre deux mix ?

---

# Exercice 4 — Planification à trois nœuds

## Objectif

Étendre le modèle à plusieurs régions reliées par des interconnexions. Les régions ont des profils de demande et de production renouvelable distincts. Le modèle peut investir dans la production, le stockage et les capacités de transport, afin de profiter de la complémentarité géographique.

On reprend la structure de `3-node-cem.ipynb` : Victoria (VIC), South Australia (SA) et New South Wales (NSW). Les données temporelles sont lues directement depuis les fichiers d’exemple hébergés par TU Berlin.

### 4.1 Charger les données et créer les nœuds

In [ ]:
import numpy as np
from pypsa.costs import annuity

RESOLUTION = 3
n_3 = pypsa.Network()
REGIONS = pd.Index(["VIC", "SA", "NSW"])
LAT = [-37.81, -34.93, -33.87]
LON = [145.01, 138.63, 151.20]

### DEBUT SOLUTION
n_3.add("Bus", REGIONS, x=LON, y=LAT)
url = "https://tubcloud.tu-berlin.de/s/oPQbAebrciFBZP2/download/australia-example-p_max_pu.csv"
p_max_pu = pd.read_csv(url, index_col=0, parse_dates=True, header=[0, 1])
url = "https://tubcloud.tu-berlin.de/s/8C6d7z3HxE7yZi9/download/australia-example-p_set.csv"
p_set = pd.read_csv(url, index_col=0, parse_dates=True)
n_3.set_snapshots(p_max_pu.index)
n_3.add("Load", REGIONS, suffix=" load", bus=REGIONS, p_set=p_set, carrier="load")
### FIN SOLUTION


### 4.2 Ajouter la production renouvelable et le délestage

Les facteurs de capacité `p_max_pu` bornent la production éolienne et solaire. Le générateur de délestage représente une solution de dernier recours, avec un coût très élevé : si le modèle l’utilise, cela signale qu’il est moins coûteux que les options disponibles ou que les contraintes imposent de ne pas satisfaire toute la demande.

In [ ]:
CARRIERS = {
    "solar": "gold", "wind": "steelblue", "load shedding": "indianred",
    "battery charger": "lightgreen", "battery discharger": "lightgreen",
    "battery storage": "grey", "electrolysis": "violet", "turbine": "violet",
    "hydrogen storage": "orchid", "hydrogen": "orchid", "AC": "black",
    "HVDC": "lightseagreen", "load": "slategrey",
}
### DEBUT SOLUTION
n_3.add("Carrier", list(CARRIERS), color=list(CARRIERS.values()))
p_wind = p_max_pu.xs("onwind", level=1, axis=1).rename(columns=lambda s: s + " wind")
p_solar = p_max_pu.xs("solar", level=1, axis=1).rename(columns=lambda s: s + " solar")
n_3.add(
    "Generator", p_wind.columns, bus=REGIONS, p_max_pu=p_wind,
    p_nom_extendable=True, capital_cost=annuity(0.05, 30) * 2_000_000,
    marginal_cost=0.5, carrier="wind",
)
n_3.add(
    "Generator", p_solar.columns, bus=REGIONS, p_max_pu=p_solar,
    p_nom_extendable=True, capital_cost=annuity(0.05, 25) * 700_000,
    marginal_cost=0.01, carrier="solar",
)
n_3.add(
    "Generator", REGIONS, suffix=" load shedding", bus=REGIONS,
    p_nom=p_set.max(), marginal_cost=3000, carrier="load shedding",
)
n_3.set_snapshots(n_3.snapshots[::RESOLUTION])
n_3.snapshot_weightings.loc[:, :] = RESOLUTION
### FIN SOLUTION


### 4.3 Premier calcul : régions isolées

On résout d’abord le modèle sans interconnexions. Ce cas sert de référence : chaque région doit équilibrer localement son offre et sa demande.

In [ ]:
### DEBUT SOLUTION
n_3.optimize()
display(n_3.statistics.energy_balance().div(1e6).round(2).sort_values())
### FIN SOLUTION


### 4.4 Ajouter du stockage et les interconnexions

Les interconnexions sont représentées par des liens HVDC bidirectionnels et sans pertes. Pour une batterie dont le chargeur et le déchargeur sont modélisés séparément, on ajoute une contrainte afin de relier leurs puissances nominales.

In [ ]:
### DEBUT SOLUTION
n_3.add("Bus", REGIONS, suffix=" hydrogen", carrier="hydrogen", x=LON, y=LAT)
n_3.add(
    "Link", REGIONS, suffix=" electrolysis", bus0=REGIONS,
    bus1=pd.Index(REGIONS) + " hydrogen", carrier="electrolysis",
    p_nom_extendable=True, efficiency=0.7,
    capital_cost=annuity(0.05, 25) * 2_500_000,
)
n_3.add(
    "Link", REGIONS, suffix=" turbine", bus0=pd.Index(REGIONS) + " hydrogen",
    bus1=REGIONS, carrier="turbine", p_nom_extendable=True,
    efficiency=0.5, capital_cost=annuity(0.05, 25) * 1_000_000,
)
n_3.add(
    "Store", REGIONS, suffix=" hydrogen storage",
    bus=pd.Index(REGIONS) + " hydrogen", carrier="hydrogen storage",
    e_nom_extendable=True, e_cyclic=True,
    capital_cost=annuity(0.05, 30) * 80,
)
n_3.add("Bus", REGIONS, suffix=" battery", carrier="battery", x=LON, y=LAT)
n_3.add(
    "Link", REGIONS, suffix=" battery charger",
    bus0=REGIONS, bus1=REGIONS + " battery", carrier="battery charger",
    p_nom_extendable=True, efficiency=0.95,
    capital_cost=annuity(0.05, 10) * 300_000,
)
n_3.add(
    "Link", REGIONS, suffix=" battery discharger",
    bus0=REGIONS + " battery", bus1=REGIONS, carrier="battery discharger",
    p_nom_extendable=True, efficiency=0.95,
)
n_3.add(
    "Store", REGIONS, suffix=" battery storage",
    bus=REGIONS + " battery", carrier="battery storage",
    e_nom_extendable=True, e_cyclic=True,
    capital_cost=annuity(0.05, 25) * 250_000,
)
connections = [("VIC", "NSW", 700), ("VIC", "SA", 650), ("NSW", "SA", 1150)]
for bus0, bus1, length in connections:
    n_3.add(
        "Link", f"{bus0}-{bus1}", bus0=bus0, bus1=bus1, carrier="HVDC",
        p_nom_extendable=True, capital_cost=annuity(0.05, 40) * 1_000 * length,
        p_min_pu=-1,
    )
### FIN SOLUTION


### 4.5 Coupler les puissances de charge et de décharge de la batterie

La contrainte supplémentaire évite que le modèle choisisse indépendamment deux puissances nominales qui ne correspondent pas au même onduleur. Le code ci-dessous suit le principe du notebook d’origine.

In [ ]:
### DEBUT SOLUTION
def battery_constraint(network, snapshots):
    dischargers = network.links.index[network.links.index.str.contains(" discharger")]
    chargers = network.links.index[network.links.index.str.contains(" charger")]
    if len(chargers) == 0 or len(dischargers) == 0:
        return
    efficiency = network.links.efficiency.loc[dischargers].values
    lhs = network.model["Link-p_nom"].loc[chargers]
    rhs = network.model["Link-p_nom"].loc[dischargers] * efficiency
    network.model.add_constraints(lhs == rhs, name="battery_inverter_ratio")

n_3.optimize(extra_functionality=battery_constraint)
display(n_3.links.query("carrier == 'HVDC'").p_nom_opt)
display(n_3.statistics.energy_balance(bus_carrier="AC").div(1e6).round(2).sort_values())
### FIN SOLUTION


**Questions de discussion**
- Pourquoi une interconnexion peut-elle réduire le coût total même si elle a un coût d’investissement ?
- Comment interpréter une capacité d’interconnexion optimale élevée ?
- Quelle différence entre un lien HVDC contrôlable et une ligne AC soumise aux lois de Kirchhoff ?


### 4.6 Pour explorer : prix régionaux, rente de congestion et valeur du réseau

#### 4.6.a Les prix par région

Avec des interconnexions bidirectionnelles sans pertes, les prix de deux régions **s'égalisent** tant que le lien qui les relie n'est pas saturé.

In [ ]:
# TODO : extraire les prix des 3 régions, calculer le prix moyen par région et la part du temps où les trois prix sont égaux
# (écart max < 0,01 €/MWh), puis tracer les prix sur une semaine.
### DEBUT SOLUTION
prix_reg = n_3.buses_t.marginal_price[list(REGIONS)]
poids3 = n_3.snapshot_weightings.generators

print("Prix moyen par région (€/MWh) :")
display(prix_reg.mul(poids3, axis=0).sum().div(poids3.sum()).round(1))

ecart_max = prix_reg.max(axis=1) - prix_reg.min(axis=1)
print(f"Part du temps avec prix identiques : {100 * poids3[ecart_max < 0.01].sum() / poids3.sum():.1f} %")

prix_reg.iloc[:56].plot(figsize=(9, 3.5), ylabel="€/MWh", title="Prix régionaux, première semaine")
plt.show()
### FIN SOLUTION

**Éléments de réponse** *(cellule de correction)*

Les prix sont égaux une grande partie du temps (le réseau n'est pas saturé), et divergent quand un lien **atteint sa capacité** : la région exportatrice a alors un prix bas (surplus local d'énergie bon marché) et la région importatrice un prix élevé. L'écart de prix est la **valeur marginale d'un MW de capacité de transport supplémentaire**.

#### 4.6.b La rente de congestion finance le réseau

Pour un lien dont la capacité est optimisée, la **rente de congestion** annuelle $\sum_t |p_t| \times |\Delta\lambda_t|$ (flux × écart de prix entre les deux extrémités) doit être égale à son **coût d'investissement annualisé**. C'est l'équivalent, pour le transport, de la condition de profit nul vue à l'exercice 2.

In [ ]:
# TODO : pour chaque lien HVDC, comparer la rente de congestion annuelle à son coût d'investissement annualisé.
### DEBUT SOLUTION
hvdc = n_3.links[n_3.links.carrier == "HVDC"]
lignes = {}
for nom, lien in hvdc.iterrows():
    flux = n_3.links_t.p0[nom]                                       # MW de bus0 vers bus1
    delta_prix = prix_reg[lien.bus1] - prix_reg[lien.bus0]
    rente = (flux * delta_prix * poids3).sum()
    lignes[nom] = {
        "Capacité optimale (MW)": lien.p_nom_opt,
        "Rente de congestion": rente,
        "Coût d'investissement": lien.p_nom_opt * lien.capital_cost,
    }
tableau_rente = pd.DataFrame(lignes).T
tableau_rente["Rente / coût"] = tableau_rente["Rente de congestion"] / tableau_rente["Coût d'investissement"]
display(tableau_rente.round(2))
### FIN SOLUTION

**Éléments de réponse** *(cellule de correction)*

Le rapport rente/coût est proche de **1** pour les liens construits : à l'optimum, un lien se paie exactement avec les différences de prix qu'il permet d'exploiter. Si la rente était supérieure au coût, on aurait intérêt à augmenter la capacité ; si elle était inférieure, on la diminuerait. Un lien de capacité nulle n'a, lui, ni rente ni coût. C'est une application directe de la dualité (conditions de complémentarité).

#### 4.6.c Quelle est la valeur du réseau ?

On fait varier le **coût d'investissement des liens HVDC** : gratuit (×0), triple (×3), ou bien **pas de réseau du tout**. On compare le coût total du système et les capacités installées.

In [ ]:
def resoudre_variante_reseau(n_ref, facteur=1.0, sans_reseau=False):
    m = copier_reseau(n_ref)
    liens = m.links.index[m.links.carrier == "HVDC"]
    if sans_reseau:
        m.links.loc[liens, "p_nom_extendable"] = False
        m.links.loc[liens, "p_nom"] = 0
    else:
        m.links.loc[liens, "capital_cost"] *= facteur
    m.optimize(extra_functionality=battery_constraint)
    return m

In [ ]:
# TODO : résoudre les variantes (référence, réseau ×0, ×3, sans réseau) et comparer coût total, capacité HVDC et capacités ENR.
### DEBUT SOLUTION
variantes = {"référence": n_3}
variantes["réseau gratuit (×0)"] = resoudre_variante_reseau(n_3, facteur=0.0)
variantes["réseau ×3"] = resoudre_variante_reseau(n_3, facteur=3.0)
variantes["sans réseau"] = resoudre_variante_reseau(n_3, sans_reseau=True)

lignes = {}
for nom, m in variantes.items():
    gen = m.generators.groupby("carrier").p_nom_opt.sum()
    lignes[nom] = {
        "Coût total (G€)": (m.statistics.capex().sum() + m.statistics.opex().sum()) / 1e9,
        "HVDC (GW)": m.links[m.links.carrier == "HVDC"].p_nom_opt.sum() / 1e3,
        "Éolien (GW)": gen.get("wind", 0) / 1e3,
        "Solaire (GW)": gen.get("solar", 0) / 1e3,
    }
display(pd.DataFrame(lignes).T.round(2))
### FIN SOLUTION

**Éléments de réponse** *(cellule de correction)*

- Le coût total est **croissant avec le coût du réseau**, et la différence entre « réseau gratuit » et « sans réseau » mesure la **valeur maximale** de l'interconnexion pour ce système.
- Moins le réseau est disponible, plus il faut **dupliquer** production et stockage localement : les capacités ENR et de stockage augmentent.
- Si le coût réel d'un lien est inférieur à sa valeur, l'investissement est rentable : c'est le raisonnement coût-bénéfice des plans de développement de réseau (ex. ENTSO-E TYNDP).

---

# Exercice 5 — Planification myope

## Objectif

Exercice de synthèse sur plusieurs périodes. Étudier une trajectoire d’investissement sur plusieurs périodes, en optimisant **une période à la fois**. À chaque étape, les capacités déjà construites sont figées ; les décisions ne sont donc pas prises en anticipant parfaitement toutes les périodes futures. C’est la différence centrale avec une optimisation intégrée de toute la trajectoire.

On reprend le modèle de planification à un nœud, puis on lui ajoute des périodes d’investissement, des actifs conventionnels et des contraintes d’émissions. Cette partie suit la logique du notebook `myopic-pathway.ipynb` fourni.

### 5.1 Construire le modèle multi-périodes

Les investissements ont lieu en 2025, 2035 et 2045. Pour garder le notebook relativement direct, on utilise le réseau `model_energy` comme socle, puis on ajoute des attributs de période. Le détail des hypothèses de coûts et d’émissions est volontairement explicite ci-dessous.

In [ ]:
from pypsa.costs import annuity

temp = pypsa.examples.model_energy()
temp.remove("Generator", "load shedding")
n_myopic = temp.copy()
INVESTMENT_PERIODS = [2025, 2035, 2045]

# Capacités préexistantes : 5 GW de solaire et d'éolien, construits en 2015.
n_myopic.generators.loc[:, ["build_year", "p_nom", "p_nom_extendable"]] = 2015, 5_000, False

# Les composants de stockage sont recréés pour chaque période d'investissement.
n_myopic.remove("Link", n_myopic.links.index)
n_myopic.remove("StorageUnit", n_myopic.storage_units.index)
n_myopic.remove("Store", n_myopic.stores.index)

for year in INVESTMENT_PERIODS:
    n_myopic.add(
        "Generator",
        temp.generators.index + f"-{year}",
        build_year=year,
        p_max_pu=temp.generators_t.p_max_pu.rename(
            columns=lambda s: s + f"-{year}"
        ).loc[:, ::-1],
        **temp.generators.rename(index=lambda s: s + f"-{year}").drop(
            columns=["build_year", "p_max_pu"]
        ),
    )
    for component in ["Link", "StorageUnit", "Store"]:
        table = temp.components[component].static
        if len(table):
            n_myopic.add(
                component, table.index, suffix=f"-{year}", build_year=year,
                **table.rename(index=lambda s: s + f"-{year}").drop(columns=["build_year"]),
            )

# Hypothèses de progrès technique au fil du temps.
n_myopic.generators.loc["solar-2035", "capital_cost"] *= 0.9
n_myopic.generators.loc["solar-2045", "capital_cost"] *= 0.85
if "battery storage-2035" in n_myopic.storage_units.index:
    n_myopic.storage_units.loc["battery storage-2035", "capital_cost"] *= 0.9
    n_myopic.storage_units.loc["battery storage-2045", "capital_cost"] *= 0.85
n_myopic.generators_t.p_max_pu.loc[:, "wind-2035"] *= 1.1
n_myopic.generators_t.p_max_pu.loc[:, "wind-2045"] *= 1.15

### 5.2 Ajouter des actifs conventionnels et une contrainte climatique

On représente une centrale au charbon existante, une centrale au gaz existante et une possibilité d’investissement dans du gaz en 2035. Les émissions sont exprimées en tonnes de CO₂ par MWh de combustible consommé ; PyPSA utilise le rendement pour relier combustible et production électrique.

In [ ]:
### DEBUT SOLUTION
n_myopic.add(
    "Carrier",
    ["coal", "gas"],
    color=["gray", "lightcoral"],
    co2_emissions=[0.336, 0.198],
)
n_myopic.add(
    "Generator", "coal", bus="electricity", carrier="coal",
    p_nom=4000, p_nom_extendable=False, build_year=1990, lifetime=40,
    capital_cost=annuity(0.07, 40) * 4_000_000,
    efficiency=0.35, marginal_cost=30,
)
n_myopic.add(
    "Generator", "gas", bus="electricity", carrier="gas",
    p_nom=6000, p_nom_extendable=False, build_year=2005, lifetime=35,
    capital_cost=annuity(0.07, 35) * 800_000,
    efficiency=0.50, marginal_cost=60,
)
n_myopic.add(
    "Generator", "gas-2035", bus="electricity", carrier="gas",
    p_nom_extendable=True, build_year=2035, lifetime=35,
    capital_cost=annuity(0.07, 35) * 900_000,
    efficiency=0.55, marginal_cost=55,
)
n_myopic.set_investment_periods(INVESTMENT_PERIODS)
n_myopic.investment_period_weightings *= 10

REDUCTION_PATH = [300e6, 100e6, 20e6]
for year, limit in zip(INVESTMENT_PERIODS, REDUCTION_PATH):
    n_myopic.add(
        "GlobalConstraint", f"co2-limit-{year}",
        type="primary_energy", carrier_attribute="co2_emissions",
        investment_period=year, constant=limit, sense="<=",
    )

# Copie du réseau (avant toute optimisation) pour la comparaison avec la planification intégrée (§5.5.c)
n_integre = n_myopic.copy()
### FIN SOLUTION

### 5.3 Optimiser période par période

La fonction `optimize_period` sélectionne les snapshots de la période considérée. Après résolution, `freeze_period` fixe les capacités qui viennent d’être décidées. Les capacités construites auparavant restent disponibles, tandis que les investissements futurs peuvent être optimisés lors des périodes suivantes.

In [ ]:
### DEBUT SOLUTION
def optimize_period(network, period):
    snapshots = network.snapshots[
        network.snapshots.get_level_values("period") == period
    ]
    return network.optimize(multi_investment_periods=True, snapshots=snapshots)

def freeze_period(network, period):
    for component in network.components[["Generator", "Link", "StorageUnit", "Store"]]:
        attr = "e_nom" if component.name == "Store" else "p_nom"
        if len(component.df):
            component.static[attr] = component.static[attr + "_opt"]
            component.static.loc[
                component.static.build_year == period, attr + "_extendable"
            ] = False

optimize_period(n_myopic, 2025)
freeze_period(n_myopic, 2025)
optimize_period(n_myopic, 2035)
freeze_period(n_myopic, 2035)
optimize_period(n_myopic, 2045)
### FIN SOLUTION


### 5.4 Lire la trajectoire

Les tableaux ci-dessous donnent un aperçu des capacités et des bilans énergétiques. Les résultats dépendent des versions de PyPSA et des données téléchargées.

In [ ]:
### DEBUT SOLUTION
display(n_myopic.generators[["carrier", "build_year", "p_nom", "p_nom_opt"]])
display(n_myopic.statistics.energy_balance().div(1e6).round(2))
display(n_myopic.global_constraints)
### FIN SOLUTION


**Questions de discussion**
- Pourquoi une optimisation myope peut-elle conserver ou construire des actifs qui deviennent peu utilisés ensuite ?
- Que changerait une optimisation intégrée de toutes les périodes ?
- Comment les objectifs d’émissions modifient-ils les coûts et le mix électrique ?

### 5.5 Pour explorer : émissions, prix du carbone, actifs sous-utilisés et valeur de l'anticipation

#### 5.5.a Vérifier la contrainte climatique et lire le prix implicite du CO₂

Chaque contrainte de plafond d'émissions a une **variable duale** : le prix implicite du carbone ($€/\text{tCO}_2$) qui rendrait la contrainte naturellement respectée. La fonction suivante calcule les émissions réelles de chaque période (tCO₂ = production / rendement × facteur d'émission).

In [ ]:
def emissions_par_periode(n):
    """Émissions annuelles de CO2 par période d'investissement (tCO2)."""
    co2 = n.carriers.co2_emissions
    gen = n.generators
    primaire = n.generators_t.p.div(gen.efficiency, axis=1)                 # MWh de combustible par pas de temps
    facteur = gen.carrier.map(co2).fillna(0)
    em = primaire.mul(facteur, axis=1).mul(n.snapshot_weightings.generators, axis=0).sum(axis=1)
    return em.groupby(level="period").sum()

In [ ]:
# TODO : comparer émissions réelles et plafonds par période, et afficher le prix implicite (mu) de chaque contrainte.
### DEBUT SOLUTION
tableau_co2 = pd.DataFrame({
    "Plafond (tCO2)": REDUCTION_PATH,
    "Émissions réelles (tCO2)": emissions_par_periode(n_myopic).reindex(INVESTMENT_PERIODS).values,
    "Prix implicite du CO2 (€/t)": -n_myopic.global_constraints["mu"].values,   # mu est négatif pour une contrainte « ≤ » active
}, index=INVESTMENT_PERIODS)
display(tableau_co2.round(1))
### FIN SOLUTION

**Éléments de réponse** *(cellule de correction)*

- Lorsque la contrainte est **active**, les émissions sont égales au plafond et le prix implicite est strictement positif ; si elle n'est **pas active** (le plafond est plus lâche que ce que l'optimisation choisit naturellement), le prix implicite vaut 0. Regardez d'abord laquelle de ces deux situations s'applique à chaque période.
- Quand les plafonds se durcissent au fil du temps et deviennent actifs, le prix implicite du carbone **augmente** : c'est le coût marginal d'abattement croissant, cohérent avec les trajectoires de décarbonation.
- Convention de signe : PyPSA renvoie `mu` négatif pour une contrainte de type « ≤ » (une unité de plafond en plus *diminue* le coût), d'où le signe moins dans le code.
- Attention : en optimisation myope, chaque période est optimisée séparément, donc le prix implicite d'une période ne tient pas compte de ce qui se passera après.

#### 5.5.b Qui produit ? Évolution de la production par technologie et des actifs sous-utilisés

In [ ]:
# TODO : calculer la production annuelle (TWh) par technologie et par période, puis le facteur de charge des centrales
# fossiles existantes (charbon 1990, gaz 2005) à chaque période.
### DEBUT SOLUTION
poids_gen = n_myopic.snapshot_weightings.generators
prod = n_myopic.generators_t.p.mul(poids_gen, axis=0).groupby(level="period").sum()
prod_par_carrier = prod.T.groupby(n_myopic.generators.carrier).sum().T / 1e6
display(prod_par_carrier.round(2))
prod_par_carrier.plot.bar(stacked=True, figsize=(7, 3.5), ylabel="TWh / an", rot=0, title="Production par technologie et par période")
plt.show()

duree_periode = poids_gen.groupby(level="period").sum()
for nom in ["coal", "gas"]:
    fc = prod[nom] / (n_myopic.generators.loc[nom, "p_nom"] * duree_periode)
    print(f"Facteur de charge de la centrale {nom} :", fc.round(2).to_dict())
### FIN SOLUTION

**Éléments de réponse** *(cellule de correction)*

Les centrales fossiles existantes voient leur **facteur de charge diminuer** quand le plafond de CO₂ se durcit, bien qu'elles soient toujours « disponibles » : elles deviennent des **actifs échoués** (*stranded assets*), construits pour un système qui n'existe plus. C'est un enjeu majeur de la transition : coût financier de ces actifs, fermetures anticipées, compensation éventuelle.

#### 5.5.c Que vaut l'anticipation ? Optimisation myope vs intégrée

La copie `n_integre`, faite avant toute optimisation (cellule 5.2), permet de résoudre **toute la trajectoire d'un coup** : le modèle connaît alors les contraintes futures et peut investir en conséquence. Comparez les investissements par technologie et année de construction.

> ⏱️ Ce calcul est plus lourd que les trois calculs successifs (il résout les trois périodes simultanément).

In [ ]:
# TODO : optimiser n_integre sur l'ensemble des périodes, puis comparer capacités (par carrier et année de construction)
# et émissions avec la planification myope.
### DEBUT SOLUTION
n_integre.optimize(multi_investment_periods=True)

def capacites(n):
    return n.generators.groupby(["carrier", "build_year"]).p_nom_opt.sum().div(1e3).round(2)

comparaison = pd.concat({"myope (GW)": capacites(n_myopic), "intégrée (GW)": capacites(n_integre)}, axis=1).fillna(0)
display(comparaison)

emis = pd.DataFrame({
    "myope": emissions_par_periode(n_myopic),
    "intégrée": emissions_par_periode(n_integre),
}) / 1e6
display(emis.round(1).rename_axis("MtCO2/an"))
### FIN SOLUTION

**Éléments de réponse** *(cellule de correction)*

En planification intégrée, le modèle **anticipe** les plafonds futurs : il peut par exemple investir plus tôt dans les technologies bas-carbone pour éviter de construire des actifs fossiles qui seraient sous-utilisés plus tard. La différence entre les deux solutions mesure la **valeur de l'anticipation** (*value of foresight*). Elle reste souvent modeste si les technologies sont bien connues et bon marché, mais devient importante quand les durées de vie des actifs sont longues et que les contraintes se durcissent vite.

Dans la réalité, aucun planificateur ne connaît parfaitement le futur : la planification myope et la planification intégrée sont deux **bornes** du comportement réel.

---

# Synthèse

| Étude de cas | Décision principale | Question à retenir |
|---|---|---|
| Un nœud | Production et stockage | Quel mix minimise le coût ? Les prix couvrent-ils les coûts ? |
| Courbes de dépistage | Parc de production optimal | Pourquoi base = fort coût fixe, pointe = fort coût variable ? Que vaut la rareté ? |
| MGA | Solution alternative sous contrainte de coût | L’optimum économique est-il unique ? |
| Trois nœuds | Production, stockage et transport | Quelle valeur a l’interconnexion géographique ? |
| Planification myope | Investissements par période | Que perd-on sans anticipation parfaite ? |

## Pour aller plus loin
- Modifier les coûts d’investissement ou les coûts marginaux et comparer les capacités optimales.
- Examiner les heures où le prix marginal est élevé et identifier la technologie marginale.
- Ajouter une technologie bas-carbone à l’exercice 2 et refaire le balayage du prix du carbone.
- Comparer la planification myope à une optimisation multi-périodes intégrée avec un plafond de CO₂ plus strict.